In [ ]:
# Import Libraries
%pip install catboost
import kagglehub
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

df_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(df_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
print(f"Before: {df.shape}")

df = df.drop('Order_ID', axis=1)
print(f"After dropping the column: {df.shape}")

In [ ]:
# Task 2: Write your code here:
missing_values = df.isnull().sum()
print("Columns with missing values:")
print(missing_values[missing_values > 0])
# Fill catogorical columns missing values
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df[col] = df[col].fillna(df[col].mode()[0])

# Fill neumerical columns missing values
for col in ['Courier_Experience_yrs', 'Delivery_Time']:
    df[col] = df[col].fillna(df[col].mean())

# Check if any missing value remains
print("Missing values remaining:", df.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(duplicates)
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)
# Check if any duplicates value remains
print("Duplicates values remaining:",  df.duplicated().sum())

In [ ]:
# Task 4: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols))

oe = OneHotEncoder(sparse_output=False)
encoded = oe.fit_transform(df[categorical_cols])
# Convert the encoded data into a DataFrame with proper column names
encoded_df = pd.DataFrame(
    encoded,
    columns=oe.get_feature_names_out(categorical_cols),
    index=df.index
)
# Drop the original categorical columns
df = df.drop(columns=categorical_cols)

# Concatenate the encoded columns back into the DataFrame
df = pd.concat([df, encoded_df], axis=1)

# Check the result
df.head()

In [ ]:
# Task 5: Write your code here:
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET, that's why we drop it
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 6: Write your code here:
# 1. What does our target variable (charges) look like?
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")
# cuz the most data is in the middle, it's normally distributed

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float) # for safety
y = df['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
X_train, X_test, y_train, y_test =  train_test_split(X, y, test_size=0.2, random_state=42)# YOUR CODE HERE

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

all_results = {'mae': []}
# Initialize and train model
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1) # n_estimators=100 = 100 decision Trees

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]
  # Train
  model.fit(X_train, y_train)
  # Predict
  y_pred = model.predict(X_test)
  # Calculate metrics
  mae = mean_absolute_error(y_test, y_pred)
  # Store results
  all_results["mae"].append(mae)

print("\nModel trained!")

print(f" Average MAE:  {np.mean(all_results['mae']):.4f}")


In [ ]:
# Task 1: Write your code here:
feature_cols = [col for col in df.columns if col != 'Delivery_Time']

# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
})
importance = importance.sort_values('importance', ascending=True).tail(10)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'])
plt.title('Top Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
valid_pred = model.predict(X_test)
plt.figure(figsize=(10, 6))
plt.hist(valid_pred, bins=30, edgecolor='black')
plt.title('Distribution of Predictions')
plt.xlabel('Predicted Delivery Time')
plt.ylabel('Count')
plt.show()

In [ ]:
# Task Bonus: Write your code here:
models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "CatBoost": CatBoostRegressor(verbose=0)
}
all_results = {}

for name in models:
  all_results[name] = {'mae': []}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)
    # Store results
    all_results[model_name]["mae"].append(mae)

for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MSE:  {np.mean(all_results[model_name]['mae']):.4f}")
